# SPATIAL INTELLIGENCE — PART 3
## Homework 02 — ETM

## 1. Import the needed libraries

In [ ]:
from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Wire import Wire
from topologicpy.Face import Face
from topologicpy.Shell import Shell
from topologicpy.Cell import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Helper import Helper
from topologicpy.Grid import Grid
from topologicpy.Graph import Graph
from topologicpy.Color import Color

## 2. Check the TopologicPy Version

In [ ]:
print("This notebook requires topologicpy version 0.9.18 or newer.")
print(Helper.Version())

## 3. Set your renderer
* Visual Studio Code: `"vscode"`
* Google Colab: `"colab"`
* Browser: `"browser"`

In [ ]:
renderer = "vscode"

## 4. Utility functions

In [ ]:
def reset_dictionaries(shell):
    faces = Topology.Faces(shell)
    for i, f in enumerate(faces):
        d = Topology.Dictionary(f)
        keys = Dictionary.Keys(d)
        for key in keys:
            if not key == 'face_id':
                d = Dictionary.RemoveKey(d, key)
        f = Topology.SetDictionary(f, d)

def transfer_dicts_by_key(topologies, selectors, key):
    dicts = {}
    for t in topologies:
        d = Topology.Dictionary(t)
        value = Dictionary.ValueAtKey(d, key, None)
        if value:
            dicts[str(value)] = t
    for s in selectors:
        d = Topology.Dictionary(s)
        value = Dictionary.ValueAtKey(d, key, None)
        if value:
            f = dicts.get(str(value), None)
            if f:
                f = Topology.SetDictionary(f, d)

## 5. Load the floor plan outline (OBJ)

In [ ]:
OBJ_PATH = r"C:\Users\etmaglari\IAAC\etmaglari_gML\Homework02\homework02_outline.obj"
objects = Topology.ByOBJPath(OBJ_PATH)
print(f"Imported {len(objects)} objects")

floor_face = None
for obj in objects:
    faces = Topology.Faces(obj)
    if faces:
        floor_face = faces[0]
        break
    wires = Topology.Wires(obj)
    if wires:
        floor_face = Face.ByWire(wires[0])
        if floor_face:
            break

print("Floor face loaded:", floor_face is not None)

b_r = Wire.BoundingRectangle(floor_face)
d_br = Topology.Dictionary(b_r)
xmin   = Dictionary.ValueAtKey(d_br, "xmin")
xmax   = Dictionary.ValueAtKey(d_br, "xmax")
ymin   = Dictionary.ValueAtKey(d_br, "ymin")
ymax   = Dictionary.ValueAtKey(d_br, "ymax")
width  = Dictionary.ValueAtKey(d_br, "width")
length = Dictionary.ValueAtKey(d_br, "length")
print(f"Bounds: x=[{xmin:.1f}, {xmax:.1f}]  y=[{ymin:.1f}, {ymax:.1f}]")
print(f"Size: {width:.1f} x {length:.1f} units")

## 6. Show the floor plan

In [ ]:
Topology.Show(floor_face,
              camera=[0, 0, 6],
              faceColor=[210, 210, 250],
              faceOpacity=1,
              edgeColor="white",
              edgeWidth=3,
              showVertices=False,
              backgroundColor="black",
              width=800, height=600,
              renderer=renderer)

## 7. Create two grids
* `grid1` — coarse vertex grid for isovist viewpoints
* `grid2` — dense edge grid for slicing the floor plan

In [ ]:
COARSE_STEP = 28
DENSE_STEP  = 1
uRange1 = list(range(0, int(width)  + COARSE_STEP, COARSE_STEP))
vRange1 = list(range(0, int(length) + COARSE_STEP, COARSE_STEP))
uRange2 = list(range(0, int(width)  + DENSE_STEP,  DENSE_STEP))
vRange2 = list(range(0, int(length) + DENSE_STEP,  DENSE_STEP))
grid1 = Grid.VerticesByDistances(floor_face, clip=True, uRange=uRange1, vRange=vRange1)
grid2 = Grid.EdgesByDistances(floor_face,    clip=True, uRange=uRange2, vRange=vRange2)

In [ ]:
Topology.Show(floor_face, grid2,
              camera=[0, 0, 6],
              faceColor=[210, 210, 250],
              faceOpacity=1,
              edgeColor="grey",
              edgeWidth=2,
              showVertices=False,
              backgroundColor="black",
              width=800, height=600,
              renderer=renderer)

## 8. Slice the floor plan with the dense grid

In [ ]:
shell = Topology.Slice(floor_face, grid2)
faces = Topology.Faces(shell)
for i, f in enumerate(faces):
    d = Dictionary.ByKeyValue("face_id", "face_"+str(i+1))
    f = Topology.SetDictionary(f, d)
print(f"Grid cells: {len(faces)}")

## 9. Derive the analysis graph and isovist viewpoints

In [ ]:
analysis_graph = Graph.ByTopology(shell)
g_verts   = Graph.Vertices(analysis_graph)
iso_verts = Topology.Vertices(grid1)
print(f"Analysis graph vertices: {len(g_verts)}")
print(f"Isovist viewpoints: {len(iso_verts)}")

## 10. Compute isovists
* Only viewpoints inside the floor boundary produce valid isovists.
* Time-consuming — expect a few minutes.

In [ ]:
isovists = []
inside_flags = Vertex.IsInternal2D(iso_verts, floor_face)
for v, is_inside in zip(iso_verts, inside_flags):
    if is_inside:
        iso = Face.Isovist(floor_face, v)
        isovists.append(iso)
    else:
        isovists.append(None)
valid_isovists = [i for i in isovists if i]
print(f"Valid isovists: {len(valid_isovists)}")

In [ ]:
Topology.Show(floor_face, valid_isovists,
              faceOpacity=0.6,
              showEdges=False,
              camera=[0, 0, 6],
              backgroundColor="black",
              width=800, height=600,
              renderer=renderer)

## 11. Compute visibility count at each isovist viewpoint
Count how many dense graph vertices fall inside each isovist.

In [ ]:
new_verts = []
n_list = []
for i, iso in enumerate(isovists):
    if iso:
        v = iso_verts[i]
        b_list = Vertex.IsInternal2D(g_verts, iso)
        b_list = [b for b in b_list if b]
        n = len(b_list)
        n_list.append(n)
        d = Dictionary.ByKeyValue("visibility", n)
        v = Topology.SetDictionary(v, d)
        new_verts.append(v)

## 12. Interpolate visibility to dense graph vertices and show

In [ ]:
for v in g_verts:
    new_v = Vertex.InterpolateValue(v, vertices=new_verts, n=2, key="visibility")

In [ ]:
minValue = min(n_list)
maxValue = max(n_list)
for v in g_verts:
    d = Topology.Dictionary(v)
    vb = Dictionary.ValueAtKey(d, "visibility")
    color = Color.AnyToHex(Color.ByValueInRange(vb, minValue=minValue, maxValue=maxValue, colorScale="thermal"))
    d = Dictionary.SetValueAtKey(d, "vb_color", color)
    d = Dictionary.SetValueAtKey(d, "size", 16)
    v = Topology.SetDictionary(v, d)

In [ ]:
reset_dictionaries(shell)
_ = transfer_dicts_by_key(faces, g_verts, "face_id")

In [ ]:
Topology.Show(faces,
              faceColorKey="vb_color",
              faceOpacity=1,
              showEdges=False,
              showVertices=False,
              camera=[0, 0, 6],
              backgroundColor="black",
              width=800, height=600,
              renderer=renderer)